# Interpreting the unified network — 2-hidden-unit GRU

Opens up the trained network as a **dynamical system**, using the standard toolkit:

- **Sussillo & Barak (2013)** — find the *fixed points* of the hidden-state update, linearise around them,
  and read stability off the Jacobian eigenvalues.
- **Mante et al. (2013)** — treat the computation as flow in a low-dimensional state space (the *phase portrait*).
- **Ji-An et al. (2025)** — the tiny-RNN framing: a network small enough that its whole state space is
  visualisable, plus *dynamical regression* of the effective input-driven update.
- **Maheswaranathan et al. (2019)** — different seeds tend to converge on the same dynamical structure, so
  the picture from one good network reflects the task rather than the initialisation.

With **2 hidden units** the state is a point in a 2D plane, so trajectories, the flow field, the fixed points
and the linear readout's decision regions can all be drawn directly with no dimensionality reduction.

Loads data from `./generated_trials_v8`.

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from scipy.optimize import minimize
from pathlib import Path

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cpu")
DATA_DIR = Path("./generated_trials_v8")
OUT_DIR = Path("./interp_v8"); OUT_DIR.mkdir(exist_ok=True)

CLASS_NAMES  = ["no det (0)", "det (1)", "right (2)", "left (3)"]
CLASS_COLORS = ["#cfcfcf", "#e0852e", "#c0392b", "#2e6ca4"]   # grey / orange / red / blue

## 2. Load data and subtask groups

In [ ]:
def load(split):
    d = np.load(DATA_DIR / f"{split}.npz", allow_pickle=True)
    out = {"X": d["X"].astype(np.float32), "y": d["y_4class"].astype(np.int64), "types": d["types"]}
    if "aud_int" in d:
        out["aud_int"] = d["aud_int"].astype(np.float32); out["vis_int"] = d["vis_int"].astype(np.float32)
    return out

train, test = load("train"), load("test")
SUBTASKS = sorted(set(test["types"]))
CONFLICT = {"det_multisensory", "loc_conflict_audL_visR", "loc_conflict_audR_visL"}
NONCONF  = [s for s in SUBTASKS if s not in CONFLICT]
print("Train:", train["X"].shape, " Test:", test["X"].shape)
print("Subtasks:", SUBTASKS)

## 3. Model

In [ ]:
class UnifiedGRU(nn.Module):
    def __init__(self, n_channels=4, hidden_size=2, n_classes=4):
        super().__init__()
        self.gru = nn.GRU(n_channels, hidden_size, batch_first=True)
        self.readout = nn.Linear(hidden_size, n_classes)
    def forward(self, x):
        x = x.transpose(1, 2)            # (B, C, T) -> (B, T, C)
        h, _ = self.gru(x)
        return self.readout(h)           # (B, T, classes)

HIDDEN = 2

## 4. Train several seeds and keep the best

A 2-unit network is sensitive to initialisation, so we train a handful of seeds and keep the one that
solves the task best (highest mean non-conflict accuracy). Per Maheswaranathan et al. (2019), the
dynamical structure should be similar across the seeds that succeed.

In [ ]:
def train_model(seed, n_epochs=50, lr=1e-3, batch=64):
    torch.manual_seed(seed); np.random.seed(seed)
    model = UnifiedGRU(4, HIDDEN, 4).to(device)
    loader = DataLoader(TensorDataset(torch.from_numpy(train["X"]), torch.from_numpy(train["y"])),
                        batch_size=batch, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr); loss_fn = nn.CrossEntropyLoss()
    for _ in range(n_epochs):
        model.train()
        for Xb, yb in loader:
            logits = model(Xb); B, T, C = logits.shape
            loss = loss_fn(logits.reshape(B*T, C), yb.unsqueeze(1).expand(B, T).reshape(B*T))
            opt.zero_grad(); loss.backward(); opt.step()
    return model

def evaluate(model):
    model.eval()
    with torch.no_grad():
        pred = model(torch.from_numpy(test["X"]))[:, -1, :].argmax(-1).numpy()
    accs = {s: float((pred[test["types"] == s] == test["y"][test["types"] == s]).mean()) for s in SUBTASKS}
    nonconf = np.mean([accs[s] for s in NONCONF])
    return pred, accs, nonconf

SEEDS = [0, 1, 2, 3, 4, 5]
best = None
for s in SEEDS:
    m = train_model(s)
    pred, accs, nc = evaluate(m)
    print("seed %d  non-conflict %.3f  overall %.3f" % (s, nc, float((pred == test["y"]).mean())))
    if best is None or nc > best[1]:
        best = (m, nc, s, pred, accs)
model, best_nc, best_seed, pred, accs = best
print("\nselected seed %d  (non-conflict %.3f)" % (best_seed, best_nc))
for s in SUBTASKS:
    print("  %-26s %.3f" % (s, accs[s]))

## 5. Hand-coded GRU update, verified against PyTorch

The flow field and fixed points need the update rule in plain numpy. We reconstruct it from the trained weights and check it reproduces PyTorch's hidden states before trusting any downstream analysis.

In [ ]:
def gru_params(model):
    g = model.gru
    Wih = g.weight_ih_l0.detach().numpy(); Whh = g.weight_hh_l0.detach().numpy()
    bih = g.bias_ih_l0.detach().numpy();   bhh = g.bias_hh_l0.detach().numpy()
    H = Whh.shape[1]; sl = lambda M, i: M[i*H:(i+1)*H]
    return dict(Wir=sl(Wih,0), Wiz=sl(Wih,1), Win=sl(Wih,2),
                Whr=sl(Whh,0), Whz=sl(Whh,1), Whn=sl(Whh,2),
                bir=bih[:H], biz=bih[H:2*H], bin_=bih[2*H:3*H],
                bhr=bhh[:H], bhz=bhh[H:2*H], bhn=bhh[2*H:3*H], H=H)

def sigmoid(v): return 1.0 / (1.0 + np.exp(-v))

def gru_step(h, x, p):
    # matches torch.nn.GRU exactly (note b_hn sits inside the reset-gated term)
    r = sigmoid(p["Wir"] @ x + p["bir"] + p["Whr"] @ h + p["bhr"])
    z = sigmoid(p["Wiz"] @ x + p["biz"] + p["Whz"] @ h + p["bhz"])
    n = np.tanh(p["Win"] @ x + p["bin_"] + r * (p["Whn"] @ h + p["bhn"]))
    return (1.0 - z) * n + z * h

P = gru_params(model)
Wro = model.readout.weight.detach().numpy()   # (4, 2)
bro = model.readout.bias.detach().numpy()     # (4,)
def readout_class(h): return int(np.argmax(Wro @ h + bro))

# --- verify on a few test sequences ---
xb = torch.from_numpy(test["X"][:8])
with torch.no_grad():
    h_pt, _ = model.gru(xb.transpose(1, 2))       # (8, 50, H)
h_pt = h_pt.numpy()
max_err = 0.0
for b in range(8):
    h = np.zeros(HIDDEN)
    for t in range(test["X"].shape[2]):
        h = gru_step(h, test["X"][b][:, t], P)
        max_err = max(max_err, np.abs(h - h_pt[b, t]).max())
print("max |manual - pytorch| hidden state =", max_err)
assert max_err < 1e-4, "manual GRU does not match PyTorch"
print("OK: hand-coded update matches PyTorch.")

## 6. The readout's decision regions

The linear readout cuts the 2D state plane into four argmax regions. Everything else is drawn on top of this, so we can see which region each trajectory and fixed point lands in.

In [ ]:
GRID = np.linspace(-1.05, 1.05, 400)
GX, GY = np.meshgrid(GRID, GRID)
flat = np.stack([GX.ravel(), GY.ravel()], 1)
region = np.argmax(flat @ Wro.T + bro, axis=1).reshape(GX.shape)

def draw_regions(ax):
    ax.pcolormesh(GX, GY, region, cmap=ListedColormap(CLASS_COLORS), alpha=0.22, shading="auto", vmin=0, vmax=3)
    ax.set_xlabel("hidden unit 1"); ax.set_ylabel("hidden unit 2")
    ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05); ax.set_aspect("equal")

fig, ax = plt.subplots(figsize=(6.2, 6))
draw_regions(ax)
import matplotlib.patches as mpatches
ax.legend(handles=[mpatches.Patch(color=CLASS_COLORS[i], label=CLASS_NAMES[i]) for i in range(4)],
          loc="upper left", fontsize=8, framealpha=0.9)
ax.set_title("Readout decision regions in 2D state space")
plt.tight_layout(); plt.show()

## 7. State-space trajectories

Run representative trials and trace the hidden state through the plane. The state starts near the origin, the stimulus (0.5–1.0 s) pushes it, and after stimulus offset it relaxes into a decision region.

In [ ]:
show_types = ["det_auditory_only", "det_visual_only", "loc_auditory_only_L", "loc_visual_only_R",
              "loc_conflict_audL_visR", "loc_conflict_audR_visL", "det_multisensory"]
fig, ax = plt.subplots(figsize=(7.5, 7))
draw_regions(ax)
T = test["X"].shape[2]; t_on, t_off = 10, 20
cmap_types = plt.cm.tab10(np.linspace(0, 1, len(show_types)))
for ti, s in enumerate(show_types):
    idx = np.where(test["types"] == s)[0][:6]   # a few example trials
    for j, i in enumerate(idx):
        h = np.zeros(HIDDEN); traj = [h.copy()]
        for t in range(T):
            h = gru_step(h, test["X"][i][:, t], P); traj.append(h.copy())
        traj = np.array(traj)
        ax.plot(traj[:, 0], traj[:, 1], color=cmap_types[ti], lw=1.0, alpha=0.7,
                label=s if j == 0 else None)
        ax.scatter(traj[t_off, 0], traj[t_off, 1], color=cmap_types[ti], s=16, zorder=3)   # stimulus offset
        ax.scatter(traj[-1, 0], traj[-1, 1], color=cmap_types[ti], s=40, marker="*", edgecolor="k", lw=0.4, zorder=4)
ax.scatter(0, 0, color="k", s=30, marker="o", zorder=5, label="start (t=0)")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8, frameon=False)
ax.set_title("Hidden-state trajectories (dot = stimulus offset, star = trial end)")
plt.tight_layout(); plt.show()

## 8. Fixed points and flow field at baseline (no input)

After the stimulus, the input is ~0 and the network must *hold* its decision. The fixed points of the zero-input map are the states the network settles into; their Jacobian eigenvalues say whether each is a stable attractor (all |eig| < 1), a saddle, or unstable.

In [ ]:
def fixed_points(x, p, inits, tol=1e-10):
    def q(h): d = gru_step(h, x, p) - h; return 0.5 * float(d @ d)
    found = []
    for h0 in inits:
        res = minimize(q, h0, method="L-BFGS-B",
                       bounds=[(-1.2, 1.2)] * p["H"], options=dict(maxiter=500))
        if res.fun < tol:
            found.append(res.x)
    # dedupe
    uniq = []
    for h in found:
        if not any(np.allclose(h, u, atol=1e-3) for u in uniq):
            uniq.append(h)
    return uniq

def jacobian(h, x, p, eps=1e-5):
    H = p["H"]; J = np.zeros((H, H)); f0 = gru_step(h, x, p)
    for i in range(H):
        hp = h.copy(); hp[i] += eps
        J[:, i] = (gru_step(hp, x, p) - f0) / eps
    return J

def classify(h, x, p):
    ev = np.linalg.eigvals(jacobian(h, x, p)); m = np.abs(ev)
    kind = "stable" if np.all(m < 1) else ("unstable" if np.all(m > 1) else "saddle")
    return kind, ev

def flow_field(ax, x, p, n=23, scale=1.0):
    gx = np.linspace(-1.0, 1.0, n)
    XX, YY = np.meshgrid(gx, gx)
    U = np.zeros_like(XX); V = np.zeros_like(YY)
    for i in range(n):
        for j in range(n):
            h = np.array([XX[i, j], YY[i, j]]); d = gru_step(h, x, p) - h
            U[i, j], V[i, j] = d
    ax.quiver(XX, YY, U, V, color="0.35", alpha=0.6, scale=scale, width=0.003)

# inits sampled from observed trajectory endpoints + random grid
ends = []
for i in range(0, len(test["X"]), 7):
    h = np.zeros(HIDDEN)
    for t in range(T): h = gru_step(h, test["X"][i][:, t], P)
    ends.append(h)
inits = ends + [np.random.uniform(-1, 1, HIDDEN) for _ in range(150)]

x0 = np.zeros(4)
fps = fixed_points(x0, P, inits)
fig, ax = plt.subplots(figsize=(7, 7))
draw_regions(ax); flow_field(ax, x0, P, scale=8)
print("Baseline fixed points:")
for h in fps:
    kind, ev = classify(h, x0, P)
    mk = {"stable": "o", "saddle": "X", "unstable": "^"}[kind]
    ax.scatter(*h, marker=mk, s=130, edgecolor="k", facecolor="white", linewidths=1.6, zorder=6)
    print("  h=[% .3f % .3f]  %-8s  class=%-10s  |eig|=%s" %
          (h[0], h[1], kind, CLASS_NAMES[readout_class(h)], np.round(np.abs(ev), 3)))
ax.set_title("Baseline (zero-input) flow field and fixed points\n(o stable, X saddle, ^ unstable)")
plt.tight_layout(); plt.show()

## 9. Input-conditioned dynamics (how the stimulus reshapes the flow)

The same network has different effective dynamics while different stimuli are on. We freeze a representative stimulus input for each context and redraw the flow field and its fixed points. This is the context-dependent computation of Mante et al. (2013), here driven implicitly by the input pattern rather than an explicit cue.

In [ ]:
I = 2.0   # representative stimulus intensity (range was 1-3)
stim_inputs = {
    "auditory only (det)":  np.array([I, I, 0, 0]),
    "visual only (det)":    np.array([0, 0, I, I]),
    "loc auditory L":       np.array([I, 0, 0, 0]),
    "loc visual R":         np.array([0, 0, 0, I]),
    "conflict audL/visR":   np.array([I, 0, 0, I]),
    "conflict audR/visL":   np.array([0, I, I, 0]),
}
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, (name, x) in zip(axes.ravel(), stim_inputs.items()):
    draw_regions(ax); flow_field(ax, x, P, scale=8)
    for h in fixed_points(x, P, inits):
        kind, _ = classify(h, x, P)
        mk = {"stable": "o", "saddle": "X", "unstable": "^"}[kind]
        ax.scatter(*h, marker=mk, s=110, edgecolor="k", facecolor="white", linewidths=1.5, zorder=6)
    ax.set_title(name, fontsize=10)
fig.suptitle("Flow field and fixed points under each stimulus (intensity = %.1f)" % I, fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.96]); plt.show()

## 10. Effective input drive (toward dynamical regression)

A first step toward Ji-An et al.'s *dynamical regression*: how each input channel displaces the state from a given point. We measure the one-step displacement caused by turning each channel on, at the origin and at a few representative states. This quantifies the 'selection vectors' (cf. Mante et al.) that each modality uses.

In [ ]:
anchor_states = {"origin": np.zeros(HIDDEN)}
for s in ["loc_auditory_only_L", "loc_visual_only_R"]:
    i = np.where(test["types"] == s)[0][0]
    h = np.zeros(HIDDEN)
    for t in range(t_off): h = gru_step(h, test["X"][i][:, t], P)   # state at stimulus offset
    anchor_states[s + " @offset"] = h

chan_names = ["aud_L", "aud_R", "vis_L", "vis_R"]
print("One-step displacement dh = F(h, channel=I) - F(h, 0)  (I = %.1f)\n" % I)
for aname, h in anchor_states.items():
    base = gru_step(h, np.zeros(4), P)
    print("anchor %-26s h=[% .3f % .3f]" % (aname, h[0], h[1]))
    for c, cn in enumerate(chan_names):
        x = np.zeros(4); x[c] = I
        dh = gru_step(h, x, P) - base
        print("    %-6s -> dh=[% .3f % .3f]  |dh|=%.3f" % (cn, dh[0], dh[1], np.linalg.norm(dh)))
    print()

## 11. How to read the figures

- **Decision regions and trajectory endpoints (sections 6–7):** each subtask's trials should settle into the
  correct class region. Detection trials land in *no-det* / *det*, localisation trials in *right* / *left*.
- **Baseline fixed points (section 8):** the stable fixed points are the network's memory states (point
  attractors for categorical decisions, Sussillo & Barak 2013). Which class region each sits in is the
  network's repertoire of held decisions, and a saddle between two attractors is the boundary the state is
  pushed across during the stimulus.
- **Input-conditioned flow (section 9):** comparing auditory-only, visual-only and conflict shows where the
  stimulus moves the attractors, and how the conflict field sits between the two unisensory fields.
  Reliability-weighting shows up as the conflict flow leaning toward the stronger cue's region.
- **Input drive (section 10):** the per-channel displacement vectors are the scaffold for full dynamical
  regression (Ji-An et al. 2025), fitting the effective update as a function of the inputs.

In [ ]:
# save the selected model for reuse
torch.save(model.state_dict(), OUT_DIR / ("unified_h2_seed%d.pt" % best_seed))
print("saved", OUT_DIR / ("unified_h2_seed%d.pt" % best_seed))